In [1]:
import math 
import pathlib
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT =  pathlib.Path.cwd()
while not  (ROOT / "llm" / "config.py").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

torch.manual_seed(0)

print("torch  :", torch.__version__)
print("repo root:",ROOT)

torch  : 2.14.1+cpu
repo root: d:\Studying\Jabarti\src\jabarti


In [2]:
from llm.config import ModelConfig

TOKENS = ["the", "cat", "sat", "on", "mat"]

cfg = ModelConfig(
    vocab_size = 32,
    d_model = 8,
    n_heads = 2,
    n_layers = 2,
    max_seq_len=16,
    dropout=0.1,
)

B=1
T = len(TOKENS)

x= torch.rand(B, T, cfg.d_model)

print("d_model    :", cfg.d_model)
print("n_heads    :", cfg.n_heads, " -> d_k =", cfg.d_k)
print("n_layers   :", cfg.n_layers)
print("d_ff       :", cfg.d_ff, " <- __post_init__ filled this in: 4 * d_model")
print("vocab_size :", cfg.vocab_size)
print("max_seq_len:", cfg.max_seq_len)

d_model    : 8
n_heads    : 2  -> d_k = 4
n_layers   : 2
d_ff       : 3072  <- __post_init__ filled this in: 4 * d_model
vocab_size : 32
max_seq_len: 16


In [3]:
import inspect, llm.model.attention as m
print(m.__file__)
print("past_len + seq_len" in inspect.getsource(m.MultiHeadAttention.forward))

d:\Studying\Jabarti\src\jabarti\llm\model\attention.py
True


In [4]:
import importlib, llm.config
importlib.reload(llm.config)
from llm.model.attention import MultiHeadAttention   # rebind the name to the new class

attn = MultiHeadAttention(cfg)                       # rebuild the module from the new class

In [5]:
from llm.model.attention import MultiHeadAttention
from llm.model.feedforward import FeedForward


attn = MultiHeadAttention(cfg)
ffn = FeedForward(cfg)

attn_out, updated_cache  = attn(x)
y = ffn(attn_out)

print("x      :", tuple(x.shape))
print("attn   :", tuple(attn_out.shape))
print("ffn    :", tuple(y.shape))
print()

print("same shape in and out?", tuple(x.shape) == tuple(y.shape))

x      : (1, 5, 8)
attn   : (1, 5, 8)
ffn    : (1, 5, 8)

same shape in and out? True


In [6]:
ln = nn.LayerNorm(cfg.d_model)

wild_input = torch.tensor([
    [
        [50.0, 52.0, 48.0, 51.0, 49.0, 53.0, 48.0, 50.0]
    ]
])

normed_input = ln(wild_input)

print(wild_input.shape)
print(normed_input.shape)


torch.Size([1, 1, 8])
torch.Size([1, 1, 8])


In [7]:
normed_input

tensor([[[-0.0739,  1.1088, -1.2567,  0.5175, -0.6653,  1.7002, -1.2567,
          -0.0739]]], grad_fn=<NativeLayerNormBackward0>)

In [8]:
print("wild_input: ")
print(
    "  mean:", round(wild_input.mean().item(), 3),
    "  std:", round(wild_input.std(unbiased=False).item(), 3)
)
print()

print("normed_input: ")
print(
    "  mean:", round(normed_input.mean().item(), 3),
    "  std:", round(normed_input.std(unbiased=False).item(), 3)
)
print()

wild_input: 
  mean: 50.125   std: 1.691

normed_input: 
  mean: -0.0   std: 1.0



## GPT

In [9]:
from llm.model.attention import MultiHeadAttention
from llm.model.feedforward import FeedForward
from llm.model.block import  TransformerBlock


attn = MultiHeadAttention(cfg)
ffn = FeedForward(cfg)

attn_out, updated_cache  = attn(x)
y = ffn(attn_out)

In [10]:
blocks = nn.ModuleList(
    TransformerBlock(cfg)
    for ix in range(cfg.n_layers)
)

In [11]:
blocks

ModuleList(
  (0-1): 2 x TransformerBlock(
    (attn): MultiHeadAttention(
      (W_q): Linear(in_features=8, out_features=8, bias=False)
      (W_k): Linear(in_features=8, out_features=8, bias=False)
      (W_v): Linear(in_features=8, out_features=8, bias=False)
      (W_o): Linear(in_features=8, out_features=8, bias=True)
      (attn_dropout): Dropout(p=0.1, inplace=False)
      (resid_dropout): Dropout(p=0.1, inplace=False)
    )
    (ffn): FeedForward(
      (fc1): Linear(in_features=8, out_features=3072, bias=True)
      (act): GELU(approximate='tanh')
      (fc2): Linear(in_features=3072, out_features=8, bias=True)
    )
    (ln1): LayerNorm((8,), eps=1e-05, elementwise_affine=True, bias=True)
    (ln2): LayerNorm((8,), eps=1e-05, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
)

In [12]:
y.shape

torch.Size([1, 5, 8])

In [13]:
ln_final = nn.LayerNorm(cfg.d_model)
lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

normed = ln_final(x)
logits = lm_head(normed)

In [14]:
logits.shape

torch.Size([1, 5, 32])

In [15]:
TOKENS = ["the", "cat", "sat", "on", "mat"]

last = logits[0,-1]
probs = F.softmax(last,dim=-1)

top = torch.topk(probs, 5)

In [16]:
top

torch.return_types.topk(
values=tensor([0.0876, 0.0807, 0.0724, 0.0686, 0.0544], grad_fn=<TopkBackward0>),
indices=tensor([ 9,  0,  3, 24,  7]))

In [17]:
top.values, top.indices

(tensor([0.0876, 0.0807, 0.0724, 0.0686, 0.0544], grad_fn=<TopkBackward0>),
 tensor([ 9,  0,  3, 24,  7]))

In [18]:
from llm.model.embeddings import InputEmbedding

embedding = InputEmbedding(cfg)

head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

print("token_embedding.weight",embedding.token_embedding.weight.shape)
print("head.weight:",head.weight.shape)


head.weight = embedding.token_embedding.weight

token_embedding.weight torch.Size([32, 8])
head.weight: torch.Size([32, 8])


In [19]:
def _init_weights(module):
    
    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, mean=0.0, std=0.2)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
            
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.2)
            
            
    def _scale_residual_projections(blocks, config):
        
        scale = math.sqrt(2*config.n_layers)
        for block in blocks:
            with torch.no_grad():
                block.attn.W_o.weight.div_(scale)
                block.ffn.fc2.weight.div_(scale)


## Loss

In [20]:
sentence = torch.tensor([[5, 9, 14, 3, 21, 7]])
inputs = sentence[:, 0:-1]
targets = sentence[:, 1:]

print("sentence:",sentence.tolist()[0])
print("inputs:",inputs.tolist()[0])
print("targets:",targets.tolist()[0])
print()
print(f"{'position':>9}{'reads':>8}{'must predict':>14}")
for t in range(inputs.size(1)):
    print(f"{t:>9}{inputs[0, t].item():>8}{targets[0, t].item():>14}")

sentence: [5, 9, 14, 3, 21, 7]
inputs: [5, 9, 14, 3, 21]
targets: [9, 14, 3, 21, 7]

 position   reads  must predict
        0       5             9
        1       9            14
        2      14             3
        3       3            21
        4      21             7


In [21]:
ln_final = nn.LayerNorm(cfg.d_model)
lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

normed = ln_final(x)
logits = lm_head(normed)

print("Logits:", tuple(logits.shape)) #(B, T, vocab_size)
print("targets:", tuple(targets.shape)) #(B, T)

flatten_logits = logits.reshape(-1, logits.size(-1))
print("flatten_logits:", tuple(flatten_logits.shape)) #(B*T, vocab_size)

flatten_targets = targets.reshape(-1)
print("flatten_targets:", tuple(flatten_targets.shape)) #(B*T, vocab_size)

loss = F.cross_entropy(flatten_logits, flatten_targets,ignore_index=cfg.pad_id)

# loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=self.config.pad_id)

Logits: (1, 5, 32)
targets: (1, 5)
flatten_logits: (5, 32)
flatten_targets: (5,)


## Ignore index

In [22]:
sentence = torch.tensor([[5, 9, 14, 3, 21, 7, cfg.pad_id, cfg.pad_id, cfg.pad_id,
                          cfg.pad_id, cfg.pad_id, cfg.pad_id]])

loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1),
                       ignore_index=cfg.pad_id)

# Validate GPT Model

In [23]:
from llm.model.gpt import GPT

my_gpt = GPT(cfg.jabarti())
print("jabarti():")
print(" total parameters  :", f"{my_gpt.num_parameters():,}")

jabarti():
 total parameters  : 50,513,408
